# Scrye — Starter Notebook

Thin entry point for interacting with the system and running experiments. Logic that proves reusable should be promoted into `src/scrye/`, not left in cells.

**Conventions**
- Import from the `scrye` package (editable install — works anywhere in the venv).
- Generated artifacts / scratch figures → `outputs/` (gitignored).
- Presentation-final figures → `docs/figures/` via `FIGURES_DIR` (tracked).
- LLM calls are cached on disk, so re-running cells is free and deterministic.

Run with the project venv kernel: `uv run jupyter lab` (or select the `.venv` kernel).

In [ ]:
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np

from scrye.config import DEFAULT_MODEL, FIGURES_DIR, OUTPUTS_DIR, OPENROUTER_MODELS
from scrye.data import load_split
from scrye.llm import LLMClient
from scrye.scoring import simbench_score, aggregate_score, bootstrap_ci

print("default model:", DEFAULT_MODEL)
print("outputs ->", OUTPUTS_DIR)
print("figures ->", FIGURES_DIR)

## 1. Load the data

In [ ]:
pop = load_split("pop")
grouped = load_split("grouped")
print(f"pop: {len(pop)} records | grouped: {len(grouped)} records")

# Inspect one conditioned record.
rec = next(r for r in grouped if not r.is_population)
print("\ndataset      :", rec.dataset_name)
print("group_prompt :", rec.group_prompt)
print("segment      :", rec.segment)
print("options      :", rec.options)
print("human_answer :", {k: round(v, 3) for k, v in rec.human_answer.items()})

## 2. A throwaway zero-shot prediction

Minimal verbalized-distribution prompt — *not* the real baseline (that belongs in `src/scrye/` once we design it), just a demonstration of the loop end to end.

In [ ]:
import json
import re

client = LLMClient(model=DEFAULT_MODEL, max_tokens=256)

def naive_predict(rec):
    """Ask the model to verbalize a probability for each option. Demo only."""
    opts = ", ".join(rec.options)
    prompt = (
        f"{rec.group_prompt}\n\n{rec.input_template}\n\n"
        f"Give the probability (0-1) that a randomly sampled such person picks each option.\n"
        f"Respond ONLY with a JSON object mapping each of [{opts}] to a probability summing to 1."
    )
    text = client.prompt(prompt)
    m = re.search(r"\{.*\}", text, re.DOTALL)
    pred = json.loads(m.group(0)) if m else {}
    return {k: float(pred.get(k, 0.0)) for k in rec.options}

pred = naive_predict(rec)
score = simbench_score(pred, rec.human_answer, options=list(rec.options))
print("prediction:", {k: round(v, 3) for k, v in pred.items()})
print("truth     :", {k: round(v, 3) for k, v in rec.human_answer.items()})
print(f"SimBench S : {score:.1f}")
print("usage     :", client.usage.as_dict())

## 3. Visualize predicted vs. empirical, and save a figure

In [ ]:
def plot_distributions(pred, truth, options, title=""):
    x = np.arange(len(options))
    w = 0.4
    fig, ax = plt.subplots(figsize=(6, 3.5))
    ax.bar(x - w / 2, [truth.get(o, 0) for o in options], w, label="human")
    ax.bar(x + w / 2, [pred.get(o, 0) for o in options], w, label="predicted")
    ax.set_xticks(x); ax.set_xticklabels(options)
    ax.set_ylabel("probability"); ax.set_title(title); ax.legend()
    fig.tight_layout()
    return fig

fig = plot_distributions(pred, rec.human_answer, rec.options,
                         title=f"{rec.dataset_name} | S={score:.1f}")
# Scratch figure -> outputs/ (gitignored). Promote to FIGURES_DIR when final.
fig.savefig(OUTPUTS_DIR / "starter_example.png", dpi=120)
plt.show()

## 4. Score a small sample (with a bootstrap CI)

Pattern for any experiment: collect per-instance scores, then report `aggregate_score` with a `bootstrap_ci`. Keep samples small while iterating — the cache makes re-runs free.

In [ ]:
sample = [r for r in grouped if r.dataset_name == "ESS"][:15]
scores = [simbench_score(naive_predict(r), r.human_answer, options=list(r.options)) for r in sample]
mean, lo, hi = bootstrap_ci(scores)
print(f"n={len(scores)}  mean S = {mean:.1f}  (95% CI {lo:.1f}, {hi:.1f})")
print("usage:", client.usage.as_dict())